In [5]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

class AntiSpoofDataset(Dataset):
    """
    Dataset لتحميل صور Live و Spoof من أي مجلد يحتوي على:
    face_dataset/
        real/
        spoof/
    """
    def __init__(self, image_paths, labels, transform=None):
        self.samples = list(zip(image_paths, labels))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(label, dtype=torch.long)
        return image, label

# تحويل الصور للنموذج
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

# المسار للمجلد الأساسي
dataset_path = r"D:\deeplearning_course\read_image_write\face_dataset"

# جمع كل الصور من مجلد real و spoof
image_paths = []
labels = []

for label_name in ["real", "spoof"]:
    class_dir = os.path.join(dataset_path, label_name)
    if not os.path.exists(class_dir):
        continue
    for fname in os.listdir(class_dir):
        if fname.lower().endswith((".png", ".jpg", ".jpeg")):
            image_paths.append(os.path.join(class_dir, fname))
            labels.append(0 if label_name=="real" else 1)

# تقسيم البيانات إلى train و valid
train_paths, valid_paths, train_labels, valid_labels = train_test_split(
    image_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

# إنشاء Dataset
train_dataset = AntiSpoofDataset(train_paths, train_labels, transform=transform)
valid_dataset = AntiSpoofDataset(valid_paths, valid_labels, transform=transform)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

print(f"عدد صور التدريب: {len(train_dataset)}")
print(f"عدد صور التحقق: {len(valid_dataset)}")

عدد صور التدريب: 203
عدد صور التحقق: 51


In [6]:
import torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"


model = models.mobilenet_v2(pretrained=True)

# Replace classifier
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features, 2
)

model = model.to(device)

c:\Users\ANASGROUP\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ANASGROUP\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [9]:
from tqdm import tqdm

epochs = 10

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()# to set the gradients of all model parameters to zero before starting backpropagation.	
        outputs = model(images) 
        loss = criterion(outputs, labels)# compute the loss	 as like corssentropy loss	
        loss.backward() # do backpropagation	 to enhnace the weights , and to make the loss smaller	
        optimizer.step() # update the weights

        train_loss += loss.item() # to accumulate the loss for the epoch	

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {train_loss:.4f} | Val Acc: {acc:.2f}%")

100%|██████████| 7/7 [00:12<00:00,  1.84s/it]


Epoch [1/10] | Loss: 1.2777 | Val Acc: 70.59%


100%|██████████| 7/7 [00:18<00:00,  2.61s/it]


Epoch [2/10] | Loss: 0.6415 | Val Acc: 86.27%


100%|██████████| 7/7 [00:28<00:00,  4.01s/it]


...

Epoch [8/10] | Loss: 0.0526 | Val Acc: 100.00%


100%|██████████| 7/7 [00:13<00:00,  1.91s/it]


Epoch [9/10] | Loss: 0.0669 | Val Acc: 100.00%


100%|██████████| 7/7 [00:14<00:00,  2.02s/it]


Epoch [10/10] | Loss: 0.0285 | Val Acc: 100.00%


In [ ]:
torch.save(model.state_dict(), "resnet50.pth")

In [ ]:
pip install onnx onnxruntime

  Using cached coloredlogs-15.0.1-py2.py3-none-any.whl.metadata (12 kB)
  Using cached humanfriendly-10.0-py2.py3-none-any.whl.metadata (9.2 kB)
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
    --------------------------------------- 0.3/16.4 MB ? eta -:--:--
    --------------------------------------- 0.3/16.4 MB ? eta -:--:--
    --------------------------------------- 0.3/16.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.4 MB 390.1 kB/s eta 0:00:41
   - -------------------------------------- 0.5/16.4 MB 390.1 kB/s eta 0:00:41
   - -------------------------------------- 0.5/16.4 MB 390.1 kB/s eta 0:00:41
   - -------------------------------------- 0.8/16.4 MB 414.1 kB/s eta 0:00:38
   - -------------------------------------- 0.8/1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.15.0 requires ml-dtypes~=0.2.0, but you have ml-dtypes 0.5.4 which is incompatible.
tensorflow-intel 2.15.0 requires numpy<2.0.0,>=1.23.5, but you have numpy 2.2.6 which is incompatible.


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# إنشاء النموذج
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)  # 2 classes

# تحميل الأوزان
model.load_state_dict(torch.load(r"D:\deeplearning_course\read_image_write\face_recognation\resnet34.pth", map_location=device))
model = model.to(device)
model.eval()


c:\Users\ANASGROUP\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ANASGROUP\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
dummy_input = torch.randn(1, 3, 224, 224).to(device)  # batch size=1, 3 channels, 224x224
torch.onnx.export(
    model, 
    dummy_input, 
    r"D:\deeplearning_course\read_image_write\face_recognation\mobilenet_2.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=11,   # مناسب لمعظم التطبيقات
    do_constant_folding=True
)

C:\Users\ANASGROUP\AppData\Local\Temp\ipykernel_11596\4263599372.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
